In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import logging
from pathlib import Path

import os
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from omegaconf import OmegaConf

from pepo.utils import constants, set_seed

OmegaConf.register_new_resolver(
    "pepo.constants",
    lambda name: getattr(constants, name),
)


In [ ]:
config_path = Path("configs").absolute()
config_name = "eval.yaml"

with initialize_config_dir(config_dir=str(config_path), version_base="1.1"):
    cfg = compose(config_name=config_name)


original_work_dir = Path.cwd()

log_level_str = cfg.get("log_level", "INFO").upper()
log_level = getattr(logging, log_level_str, logging.INFO)

logger = instantiate(
    cfg.logger,
    log_dir=str(original_work_dir / "logs"),
    level=log_level,
)

resolved_cfg = OmegaConf.to_container(cfg, resolve=True)
logger.info("PEPO Evaluation - Starting")
logger.info(f"Configuration:\n{OmegaConf.to_yaml(resolved_cfg)}")

set_seed(cfg.seed)
logger.info(f"Random seed set to: {cfg.seed}")

if not os.getenv("HF_TOKEN"):
    logger.warning("HF_TOKEN environment variable not set. Model loading may fail if models are private.")

# Instantiate managers
device_manager = instantiate(cfg.device, logger=logger)
hub_manager = instantiate(cfg.hub, logger=logger)

# Instantiate model (same as chat.py)
model = instantiate(
    cfg.model,
    logger=logger,
    device_manager=device_manager,
    hub_manager=hub_manager,
)


In [ ]:

# Instantiate evaluator
evaluator = instantiate(cfg.evaluator, logger=logger)

responses_exist = evaluator.responses_exist()
force_regenerate = cfg.get("force_regenerate", False)


In [ ]:
evaluator.dataset

In [ ]:
output_ids, output_mask = evaluator.generate_responses(model,  max_new_tokens=150, batch_size=10)

# # Evaluate responses
# logger.info("Running evaluation (implementation pending)")
# evaluator.evaluate(evaluator.responses_file)

In [ ]:
for i in range(output_ids.shape[0]):
    print(output_mask[i])
    print(output_ids[i])
    print('-' * 100)


In [ ]:
# gather all output_ids where output_mask is true

# replace tokens in output_ids with 0 where output_mask is false with tokenizer.pad_token_id
tokenizer = model.get_tokenizer()
ids = output_ids.clone()
ids = ids.where(output_mask.bool(), tokenizer.pad_token_id)
print(ids)

for i in range(output_ids.shape[0]):
    print(output_mask[i])
    print(ids[i])
    print('-' * 100)

# # collect non-zero ids to list
# ids = ids.tolist()
# tokens = []
# for l in ids:
#     tmp = []
#     for i in l:
#         if i != 0:
#             print(f"{str(i):>5}", end=" ")
#             tmp.append(i)
#     tokens.append(tmp)
#     print()

for t in ids:
    print(tokenizer.decode(t, skip_special_tokens=True))
    print('-' * 100)

# tokenizer.decode(ids, skip_special_tokens=True)








In [ ]:

tokenizer = model.get_tokenizer()
for m in range(output_ids.shape[0]):
    print([output_mask[m]])
    output = []
    for i in range(output_ids.shape[1]):
        if output_mask[m][i]:
            output.append(output_ids[m][i])
    print(tokenizer.decode(output))
    print('-' * 100)